In [ ]:
!pip install yfinance pandas numpy

In [9]:
import os
import yfinance as yf
import pandas as pd
import numpy as np

ticker = "^HSI"
df = yf.download(ticker, start="2010-01-01", end="2026-06-01")

#clean up multi-index columns if yfinance returns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Calculate he MACD (12, 26, 9)
exp1 = df['Close'].ewm(span=12, adjust=False).mean()
exp2 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = exp1 - exp2
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

# RSI (14)

delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))


# drop rows with NaN values reult from rolling windows

df.dropna(inplace=True)

#format colummns the date index

df = df.reset_index()
df.rename(columns={'Date': 'date'}, inplace=True)

df['date'] = df['date'].dt.strftime('%Y-%m-%d %H:%M:%S')

#reorder columns
feature_order = ['date', 'Open', 'High', 'Low', 'Volume', 'RSI', 'MACD', 'MACD_Signal', 'Close']
df = df[feature_order]

os.makedirs('./datasets', exist_ok=True)
csv_path = './datasets/hsi_custom.csv'
df.to_csv(csv_path, index=False)

print(f"Total rows: {df.shape[0]} trading days | Total features: {df.shape[1] - 1}")




[*********************100%***********************]  1 of 1 completed

Total rows: 4023 trading days | Total features: 8


In [10]:
verify_df = pd.read_csv('./datasets/hsi_custom.csv')

print(list(verify_df.columns))
print(verify_df.head(3))

['date', 'Open', 'High', 'Low', 'Volume', 'RSI', 'MACD', 'MACD_Signal', 'Close']
                  date          Open          High           Low      Volume  \
0  2010-01-21 00:00:00  21192.830078  21272.259766  20828.240234  2465600000   
1  2010-01-22 00:00:00  20527.199219  20738.990234  20250.359375  3220400000   
2  2010-01-25 00:00:00  20447.789062  20619.699219  20422.669922  1909800000   

         RSI        MACD  MACD_Signal         Close  
0  33.249895 -145.072551   -20.743865  20862.669922  
1  31.739128 -206.585056   -57.912103  20726.179688  
2  18.582310 -262.605592   -98.850801  20598.550781  
